# this noteboook includes implementation of different categorical encodings on a real life Heart dataset:
https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction <br>
later, the encoding is simplified and redone easily using column transformer class in sklearn.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,OrdinalEncoder,OneHotEncoder

In [24]:
heart = pd.read_csv("heart.csv")
heart.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [25]:
# in first sceanrio, we are manually concatinating encoded features, there are too many nominal,numerical and ordinal features
# for simplicity, one feature of each type is chosen.
heart = heart[['Sex','Age','ChestPainType','ST_Slope','RestingECG','HeartDisease']]

heart
# there is no any null data , so imputation is not required.

,Sex,Age,ChestPainType,ST_Slope,RestingECG,HeartDisease
0,M,40,ATA,Up,Normal,0
1,F,49,NAP,Flat,Normal,1
2,M,37,ATA,Up,ST,0
3,F,48,ASY,Flat,Normal,1
4,M,54,NAP,Up,Normal,0
...,...,...,...,...,...,...
913,M,45,TA,Flat,Normal,1
914,M,68,ASY,Flat,Normal,1
915,M,57,ASY,Flat,Normal,1
916,F,57,ATA,Flat,LVH,1


In [41]:
heart['ST_Slope'].value_counts()

ST_Slope
Flat    460
Up      395
Down     63
Name: count, dtype: int64

In [28]:
# spitting the data 
X_train,X_test,y_train,y_test = train_test_split(heart.drop(columns=['HeartDisease']),heart['HeartDisease'],test_size=0.2,random_state=1)

In [30]:
X_train

,Sex,Age,ChestPainType,ST_Slope,RestingECG
852,M,43,ASY,Flat,LVH
121,F,52,NAP,Up,Normal
664,F,65,ASY,Flat,LVH
187,M,41,ASY,Flat,Normal
108,M,50,ASY,Up,Normal
...,...,...,...,...,...
767,F,54,NAP,Up,LVH
72,M,52,ASY,Flat,Normal
908,M,63,ASY,Up,LVH
235,M,39,ATA,Flat,Normal


In [43]:
# encoding Ordinal categorical feature : 'RestingECG
oe = OrdinalEncoder(categories=[['ST','LVH','Normal']],dtype=int)
X_train_Ordinal_cols = oe.fit_transform(X_train[['RestingECG']])
X_test_Ordinal_cols = oe.transform(X_test[['RestingECG']])

In [44]:
# encoding nominal categorical features using ohe 
ohe = OneHotEncoder(drop='first',sparse_output=False,dtype=int)
X_train_nominal_cols = ohe.fit_transform(X_train[['Sex','ChestPainType','ST_Slope']])
X_test_nominal_cols = ohe.transform(X_test[['Sex','ChestPainType','ST_Slope']])
X_test_nominal_cols

array([[1, 0, 0, 0, 0, 0],
       [1, 0, 0, 0, 0, 0],
       [1, 0, 0, 0, 1, 0],
       ...,
       [0, 0, 1, 0, 0, 1],
       [1, 1, 0, 0, 0, 1],
       [1, 0, 0, 0, 1, 0]])

In [48]:
# extracting Age
X_train_age = X_train[['Age']].values
X_test_age = X_test[['Age']].values

In [ ]:
# adding all the different types of cols
X_train_transformed = np.concatenate((X_train_age,X_train_Ordinal_cols,X_train_nominal_cols),axis=1)
X_test_transformed = np.concatenate((X_test_age,X_test_Ordinal_cols,X_test_nominal_cols),axis=1)
X_train_transformed.shape

(734, 8)

# Using Column Transformer
 We can do all this without the overhead of adding and seperating columns.

In [55]:
from tkinter import Label
from sklearn.compose import ColumnTransformer
transformer = ColumnTransformer( transformers=[
    ('tfr1',OrdinalEncoder(categories=[['ST','LVH','Normal']]),['RestingECG']),
    ('tfr2',OneHotEncoder(sparse_output=False,drop='first'),['Sex','ChestPainType','ST_Slope'])], remainder='passthrough')

In [59]:
transformer.fit_transform(X_train).shape

(734, 8)

In [58]:
transformer.transform(X_test).shape

(184, 8)